# Data Prep Notebook
Takes Data from ROOT Trees and Create a .pkl file to train a neural net with

In [ ]:
import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt

Data Cleaning Function Get ROOT Branch names and iterates through a TTree to get features into a Pandas Dataframe

In [ ]:
#Uproot3
def data_cleaning_uproot3(root_data,features_list = [],cut_list=[]):
    if(len(features_list) == 0):
        print("No Training Variables Selected! ")
        return
    root_Data = uproot.open(root_data) # root file
    tree_name = root_Data.keys()[0] # tree name
    tree = root_Data[tree_name] # TTree Object
    branch_keys = tree.keys() # TBranch byte names
    data = {}
    for i in range(len(branch_keys)):
        key = branch_keys[i]
        name = key.decode('utf-8')
        if name in features_list:
            arr = tree[key].array()
            data[name] = np.array(arr)
        if name in cut_list:
            arr = tree[key].array()
            data[name] = np.array(arr)

    df = pd.DataFrame.from_dict(data)
    return df

In [ ]:
def data_cleaning(root_path, features_list=None, cut_list=None, tree_name=None):

    features_list = features_list or []
    cut_list = cut_list or []
    if not features_list and not cut_list:
        print("No variables selected (features_list and cut_list are empty).")
        raise RuntimeError("Something went wrong while loading the ROOT file")
    # Open file
    with uproot.open(root_path) as f:
        # Pick a tree
        tree = None
        if tree_name is not None:
            # Check and make sure that the structure is a TTree, fail running if not
            # The ReactionFilter produces a TTree, however a ROOT file may only consist
            # of TH1 objects(for example)
            if tree_name in f:
                obj = f[tree_name]
                if hasattr(obj, "arrays"):
                    tree = obj
                else:
                    raise ValueError(f"Object '{tree_name}' exists but is not a TTree.")
            else:
                raise KeyError(f"TTree '{tree_name}' not found in file: {root_path}")
        else:
            # Auto-detect first TTree (can be changed to check multiple trees if necessary!)
            # For our purpose we assume 1 tree in ROOT file (ReactionFilter schema)
            for k, obj in f.items():
                if hasattr(obj, "arrays"):
                    tree = obj
                    break
            if tree is None:
                raise RuntimeError("No TTree found in the file.")
        available = set(tree.keys())  # branch names
        wanted = list(dict.fromkeys(features_list + cut_list))  # keep order, ordering is important
        # in TMVA but not so much in Tensorflow as long as you keep the proper column names (branch
        # names)
        # Warn or fail on missing branches
        missing = [b for b in wanted if b not in available] #check if we got all the names of the branches
        #correct
        if missing:
            raise KeyError(f"Branches not found: {missing}\nAvailable example: {sorted(list(available))[:20]}")

        arrays = tree.arrays(wanted, library="np")
        # Build DataFrame
        df = pd.DataFrame(arrays)
        return df

In [ ]:
feature_cuts = ["BeamP4_meas_E","PositiveP4_kin_X","PositiveP4_kin_Y","PositiveP4_kin_Z","NegativeP4_kin_X","NegativeP4_kin_Y","NegativeP4_kin_Z"]

We Find the names of the ROOT Branches from the ROOT file and name the columns of the dataframe the appropriate names

In [ ]:
features = ["dEdx_FDC","TrackFCAL_DOCA","Energy_FCAL","E1E9_FCAL","E9E25_FCAL","SumU_FCAL","SumV_FCAL"]

print("Features to train on:")

training_features = []

for i in features:
    ptrack = "Positive_" + i
    ntrack = "Negative_" + i
    print(ptrack)
    print(ntrack)
    training_features.append(ptrack)
    training_features.append(ntrack)

print(training_features)

Run both files through data cleaning and assign answer values to each 0 = Pions, 1 = Kaons

In [ ]:

file_name = "/content/flat_pippim.root"
root_data = file_name
df_pions = data_cleaning(root_data,features_list=training_features,cut_list=feature_cuts)
df_pions["pi_k"] = 0

In [ ]:
file_name = "/content/flat_kpkm.root"
root_data = file_name
df_kaons = data_cleaning(root_data,features_list=training_features,cut_list=feature_cuts)
df_kaons["pi_k"] = 1

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes[0,0].hist(df_pions['Positive_dEdx_FDC'],bins=100,range=[0,0.000005], histtype='step', linewidth=5,color='black',alpha=0.9,label='Pions')
axes[0,0].hist(df_kaons['Positive_dEdx_FDC'],bins=100,range=[0,0.000005], histtype='step', linewidth=5,color='red',alpha=0.9,label='Kaons')
axes[0,0].set_title("Plus Track FDC dE/dx",fontsize=20)
axes[0,0].legend(loc="upper right")

axes[1,0].hist(df_pions['Negative_dEdx_FDC'],bins=100,range=[0,0.000005], histtype='step', linewidth=5, color='black',alpha=0.9,label='Pions')
axes[1,0].hist(df_kaons['Negative_dEdx_FDC'],bins=100,range=[0,0.000005], histtype='step', linewidth=5, color='red',alpha=0.9,label='Kaons')
axes[1,0].set_title("Minus Track FDC dE/dx",fontsize=20)
axes[1,0].legend(loc="upper right")

axes[0,1].hist(df_pions['Positive_TrackFCAL_DOCA'],bins=100,range=[0,5], histtype='step', linewidth=5, color='black',alpha=0.9,label='Pions')
axes[0,1].hist(df_kaons['Positive_TrackFCAL_DOCA'],bins=100,range=[0,5], histtype='step', linewidth=5, color='red',alpha=0.9,label='Kaons')
axes[0,1].set_title("Plus Track FCAL DOCA",fontsize=20)
axes[0,1].legend(loc="upper right")

axes[1,1].hist(df_pions['Negative_TrackFCAL_DOCA'],bins=100,range=[0,5], histtype='step', linewidth=5, color='black',alpha=0.9,label='Pions')
axes[1,1].hist(df_kaons['Negative_TrackFCAL_DOCA'],bins=100,range=[0,5], histtype='step', linewidth=5, color='red',alpha=0.9,label='Kaons')
axes[1,1].set_title("Minus Track FCAL DOCA",fontsize=20)
axes[1,1].legend(loc="upper right")

axes[0,2].hist(df_pions['Positive_Energy_FCAL'],bins=100,range=[0,1], histtype='step', linewidth=5, color='black',alpha=0.9,label='Pions')
axes[0,2].hist(df_kaons['Positive_Energy_FCAL'],bins=100,range=[0,1], histtype='step', linewidth=5, color='red',alpha=0.9,label='Kaons')
axes[0,2].set_title("Plus Track FCAL Energy",fontsize=20)
axes[0,2].legend(loc="upper right")
axes[0,2].set_yscale('log')

axes[1,2].hist(df_pions['Negative_Energy_FCAL'],bins=100,range=[0,1], histtype='step', linewidth=5, color='black',alpha=0.9,label='Pions')
axes[1,2].hist(df_kaons['Negative_Energy_FCAL'],bins=100,range=[0,1], histtype='step', linewidth=5, color='red',alpha=0.9,label='Kaons')
axes[1,2].set_title("Minus Track FCAL Energy",fontsize=20)
axes[1,2].legend(loc="upper right")
axes[1,2].set_yscale('log')
plt.savefig("set1.pdf")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes[0,0].hist(df_pions['Positive_E1E9_FCAL'],bins=100,range=[0,1.2], color='black',histtype='step',  linewidth=5, alpha=0.9,label='Pions')
axes[0,0].hist(df_kaons['Positive_E1E9_FCAL'],bins=100,range=[0,1.2], color='red',histtype='step',  linewidth=5, alpha=0.9,label='Kaons')
axes[0,0].set_title("Plus Track E1/E9",fontsize=20)
axes[0,0].legend(loc="upper right")
axes[0,0].set_yscale("log")

axes[1,0].hist(df_pions['Negative_E1E9_FCAL'],bins=100,range=[0,1.2], color='black',histtype='step',  linewidth=5, alpha=0.9,label='Pions')
axes[1,0].hist(df_kaons['Negative_E1E9_FCAL'],bins=100,range=[0,1.2], color='red',histtype='step',  linewidth=5, alpha=0.9,label='Kaons')
axes[1,0].set_title("Minus Track E1/E9",fontsize=20)
axes[1,0].legend(loc="upper right")
axes[1,0].set_yscale("log")

axes[0,1].hist(df_pions['Positive_E9E25_FCAL'],bins=100,range=[0,1.2], color='black',histtype='step',  linewidth=5, alpha=0.9,label='Pions')
axes[0,1].hist(df_kaons['Positive_E9E25_FCAL'],bins=100,range=[0,1.2], color='red',histtype='step',  linewidth=5, alpha=0.9,label='Kaons')
axes[0,1].set_title("Plus Track FCAL E9/E25",fontsize=20)
axes[0,1].legend(loc="upper right")
axes[0,1].set_yscale("log")

axes[1,1].hist(df_pions['Negative_E9E25_FCAL'],bins=100,range=[0,1.2], color='black',histtype='step',  linewidth=5, alpha=0.9,label='Pions')
axes[1,1].hist(df_kaons['Negative_E9E25_FCAL'],bins=100,range=[0,1.2], color='red',histtype='step',  linewidth=5, alpha=0.9,label='Kaons')
axes[1,1].set_title("Minus Track FCAL E9/E25",fontsize=20)
axes[1,1].legend(loc="upper right")
axes[1,1].set_yscale("log")

axes[0,2].hist(df_pions['Positive_SumU_FCAL'],bins=100,range=[0,50], color='black',histtype='step',  linewidth=5, alpha=0.9,label='Pions')
axes[0,2].hist(df_kaons['Positive_SumU_FCAL'],bins=100,range=[0,50], color='red',histtype='step',  linewidth=5, alpha=0.9,label='Kaons')
axes[0,2].set_title("Positive Sum U",fontsize=20)
axes[0,2].legend(loc="upper right")
axes[0,2].set_yscale("log")

axes[1,2].hist(df_pions['Negative_SumU_FCAL'],bins=100, range=[0,50], color='black',histtype='step',  linewidth=5, alpha=0.9,label='Pions')
axes[1,2].hist(df_kaons['Negative_SumU_FCAL'],bins=100, range=[0,50], color='red',histtype='step',  linewidth=5, alpha=0.9,label='Kaons')
axes[1,2].set_title("Minus Sum U",fontsize=20)
axes[1,2].legend(loc="upper right")
axes[1,2].set_yscale("log")

axes[0,3].hist(df_pions['Positive_SumV_FCAL'],bins=100,range=[0,50], color='black',histtype='step',  linewidth=5, alpha=0.9,label='Pions')
axes[0,3].hist(df_kaons['Positive_SumV_FCAL'],bins=100,range=[0,50], color='red',histtype='step',  linewidth=5, alpha=0.9,label='Kaons')
axes[0,3].set_title("Positive Sum V",fontsize=20)
axes[0,3].legend(loc="upper right")
axes[0,3].set_yscale("log")

axes[1,3].hist(df_pions['Negative_SumV_FCAL'],bins=100, range=[0,50], color='black',histtype='step',  linewidth=5, alpha=0.9,label='Pions')
axes[1,3].hist(df_kaons['Negative_SumV_FCAL'],bins=100, range=[0,50], color='red',histtype='step',  linewidth=5, alpha=0.9,label='Kaons')
axes[1,3].set_title("Minus Sum V",fontsize=20)
axes[1,3].legend(loc="upper right")
axes[1,3].set_yscale("log")
plt.savefig("set2.pdf")
plt.show()

Below we exlore 2 separate ways to apply cuts in Pandas

In [ ]:
df_pions_filtered = df_pions[(df_pions["BeamP4_meas_E"] < 8.8) & (df_pions["BeamP4_meas_E"] > 8.2)]
df_pions_filtered = df_pions_filtered.reset_index(drop=True)


df_kaons_filtered = df_kaons[(df_kaons["BeamP4_meas_E"] < 8.8) & (df_kaons["BeamP4_meas_E"] > 8.2)]
df_kaons_filtered = df_kaons_filtered.reset_index(drop=True)

In [ ]:
df_pions_filtered["p_pos"] = np.sqrt(
    df_pions_filtered["PositiveP4_kin_X"]**2 +
    df_pions_filtered["PositiveP4_kin_Y"]**2 +
    df_pions_filtered["PositiveP4_kin_Z"]**2
)

df_pions_filtered["p_neg"] = np.sqrt(
    df_pions_filtered["NegativeP4_kin_X"]**2 +
    df_pions_filtered["NegativeP4_kin_Y"]**2 +
    df_pions_filtered["NegativeP4_kin_Z"]**2
)

df_pions_filtered = df_pions_filtered[
    (df_pions_filtered["p_pos"].between(2, 6)) &
    (df_pions_filtered["p_neg"].between(2, 6))
]

df_pions_filtered = df_pions_filtered.drop(columns=["p_pos", "p_neg"])
df_pions_filtered = df_pions_filtered.reset_index(drop=True)


df_kaons_filtered["p_pos"] = np.sqrt(
    df_kaons_filtered["PositiveP4_kin_X"]**2 +
    df_kaons_filtered["PositiveP4_kin_Y"]**2 +
    df_kaons_filtered["PositiveP4_kin_Z"]**2
)

df_kaons_filtered["p_neg"] = np.sqrt(
    df_kaons_filtered["NegativeP4_kin_X"]**2 +
    df_kaons_filtered["NegativeP4_kin_Y"]**2 +
    df_kaons_filtered["NegativeP4_kin_Z"]**2
)

df_kaons_filtered = df_kaons_filtered[
    (df_kaons_filtered["p_pos"].between(2, 6)) &
    (df_kaons_filtered["p_neg"].between(2, 6))
]

df_kaons_filtered = df_kaons_filtered.drop(columns=["p_pos", "p_neg"])

df_kaons_filtered = df_kaons_filtered.reset_index(drop=True)

before = len(df_pions)
after = len(df_pions_filtered)
removed = before - after

print(f"Before: {before}, After: {after}, Removed: {removed}")

before = len(df_kaons)
after = len(df_kaons_filtered)
removed = before - after

print(f"Before: {before}, After: {after}, Removed: {removed}")

In [ ]:
min_rows = min(len(df_pions_filtered), len(df_kaons_filtered))

df_pions_balanced = df_pions_filtered.sample(n=min_rows, random_state=42).reset_index(drop=True)
df_kaons_balanced = df_kaons_filtered.sample(n=min_rows, random_state=42).reset_index(drop=True)

In [ ]:
final_df = pd.concat([df_pions_balanced, df_kaons_balanced], ignore_index=True)
final_df = final_df.drop(columns=["BeamP4_meas_E","PositiveP4_kin_X","PositiveP4_kin_Y","PositiveP4_kin_Z","NegativeP4_kin_X","NegativeP4_kin_Y","NegativeP4_kin_Z"])

In [ ]:
final_df = final_df.reset_index(drop=True)
print(final_df.info())
output_file_name = "training_file_kaon_pion.pkl"
output_path = output_file_name
final_df.to_pickle(output_path)